# Física BR · Taller complementario: Ajuste de Datos y Regresión
### Física BR — Física Mecánica
**Profesor:** Juan David Betancur Ríos · **Semestre:** 2026-2

---
*Física interactiva para ingeniería.* Taller complementario del capítulo 3 (Diseño experimental y análisis de datos,
secciones 3.4 a 3.6). En ingeniería casi ninguna constante se lee en una tabla: se obtiene ajustando un modelo a datos
medidos. Aquí calibrarás un sensor, reportarás una constante con su incertidumbre, descubrirás una ley potencial en un
vertedero y estimarás la constante de tiempo de enfriamiento de un transformador. En cada caso, la pregunta no es solo
«¿cuánto da?», sino «¿el modelo es el correcto y cuánto confío en el número?». Requiere **Errores e Incertidumbre**.

**Convenciones:** tres cifras significativas en los resultados · coma decimal · los datos de cada tabla son sintéticos
(generados con un modelo físico más ruido de medición) y se presentan con la resolución del instrumento.

**Ecuaciones clave:**
- Recta de mínimos cuadrados:  m = [nΣxy − ΣxΣy]/[nΣx² − (Σx)²] ,  b = ȳ − m·x̄       ·      R² = 1 − SS_res/SS_tot
- Residuales:  eᵢ = yᵢ − (m·xᵢ + b)      ·      s_y = √[Σeᵢ²/(n − 2)]
- Incertidumbres:  u(m) = s_y/√Σ(xᵢ − x̄)² ,  u(b) = s_y·√[Σxᵢ²/(n·Σ(xᵢ − x̄)²)]
- Linealización:  y = a·xⁿ → log y = log a + n·log x ;  y = a·e^(−t/τ) → ln y = ln a − t/τ

> **Uso:** `Entorno de ejecución → Ejecutar todo`. Competencias: [CL, UFT, UCM, PRC, ULC].

---
> © 2026 **Juan David Betancur Ríos** (docente de física para ingeniería) y **Nora Yamile Rojas Cataño** (aporte pedagógico y metodología de evaluación). Material educativo de uso académico (Física BR). Todos los derechos reservados.
> *Elaborado con apoyo de herramientas de inteligencia artificial, bajo la supervisión y criterio pedagógico de los autores.*


---
## Punto 1 — Calibración de un transmisor de presión de 4–20 mA (Ing. Química/Instrumentación)  ★★  [CL, UCM]
En una planta se calibra un transmisor de presión de rango **0 a 600 kPa** con salida de corriente de **4 a 20 mA**. Se
aplican presiones de referencia con un calibrador y se lee la corriente:

| P (kPa) | 0 | 100 | 200 | 300 | 400 | 500 | 600 |
|---|---|---|---|---|---|---|---|
| I (mA) | 4,09 | 6,75 | 9,35 | 12,08 | 14,71 | 17,36 | 20,06 |

**Modelo:**  I = m·P + b   (m: sensibilidad en mA/kPa; b: corriente en cero, el «cero vivo» de 4 mA)

**Uso inverso (en operación):**  P = (I − b)/m

> 🔧 **Aplicación en ingeniería:** la recta de calibración es lo que convierte la señal del instrumento en una magnitud
> física en el sistema de control. El cero en 4 mA (y no en 0 mA) permite distinguir «presión cero» de «cable roto». La
> recta solo vale dentro del rango calibrado: fuera de él, el instrumento puede saturarse o dejar de ser lineal.

> 🤖 **TIP-IA — Verificador crítico:**
> Dile a un asistente: *"Calibré un transmisor de 4–20 mA y obtuve una recta con R² muy cercano a 1. Actúa como evaluador escéptico: pregúntame qué significan físicamente la pendiente y el intercepto, en qué unidades están y si puedo usar la recta para una lectura de 25 mA. No me des la respuesta."* Primero resuelves tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

try:
    PUNTO_DE_BR
except NameError:
    PUNTO_DE_BR = {}         # id_preg -> punto (botón «Mostrar solución») al que pertenece
    SOL_ANTES_BR = set()     # preguntas cuya solución se abrió antes del primer intento
    _PUNTO_ACTUAL_BR = [0]
def mostrar_solucion(func_solucion):
    # La solución es libre. Si se abre antes de intentar alguna pregunta del punto, queda anotado en la nota (no resta).
    _PUNTO_ACTUAL_BR[0] += 1; punto = _PUNTO_ACTUAL_BR[0]
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        try: pend = [k for k, p in PUNTO_DE_BR.items() if p == punto and INTENTOS_BR.get(k, [0])[0] == 0]
        except NameError: pend = []
        SOL_ANTES_BR.update(pend)
        with salida:
            salida.clear_output()
            if pend:
                print("ℹ Abriste la solución antes de intentar " + ("una pregunta" if len(pend) == 1 else "varias preguntas") + " de este punto.")
                print("  No resta puntos: solo queda anotado en tu nota, para que sepas cuáles resolviste por tu cuenta.\n")
            func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            try: vistas = sum(1 for k in PESOS_BR if k in SOL_ANTES_BR)
            except NameError: vistas = 0
            print(f"  Solución vista antes de intentar: {vistas}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if vistas: print("  ℹ En las preguntas donde abriste la solución antes, la nota no dice si sabes resolverlas sin ayuda: vuelve a intentarlas en unos días.")
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def _ajuste(x, y):
    # Mínimos cuadrados para y = m·x + b (texto guía, Ec. 3.1 y 3.2) con incertidumbres estándar de m y b
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    m = (n*np.sum(x*y) - x.sum()*y.sum())/(n*np.sum(x*x) - x.sum()**2); b = y.mean() - m*x.mean()
    r = y - (m*x + b); R2 = 1 - np.sum(r*r)/np.sum((y - y.mean())**2)
    sy = np.sqrt(np.sum(r*r)/(n - 2)); Sxx = np.sum((x - x.mean())**2)
    return m, b, R2, sy/np.sqrt(Sxx), sy*np.sqrt(np.sum(x*x)/(n*Sxx)), r

P1_P = np.array([0, 100, 200, 300, 400, 500, 600.]); P1_I = np.array([4.09, 6.75, 9.35, 12.08, 14.71, 17.36, 20.06])

def esquema():
    m, b, R2, um, ub, r = _ajuste(P1_P, P1_I)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), gridspec_kw={'width_ratios': [1.4, 1]})
    a1.scatter(P1_P, P1_I, color='#2980b9', zorder=5, label='datos de calibración'); pp = np.linspace(-50, 800, 50)
    a1.plot(pp, m*pp + b, color='#8e44ad', lw=1.5, label='recta de ajuste'); a1.axvspan(600, 800, color='#c0392b', alpha=0.1)
    a1.text(610, 6, 'fuera del rango\ncalibrado', fontsize=8, color='#c0392b'); a1.axhline(4, ls=':', color='#7f8c8d'); a1.axhline(20, ls=':', color='#7f8c8d')
    a1.set_xlabel('presión P (kPa)'); a1.set_ylabel('corriente I (mA)'); a1.legend(fontsize=8); a1.set_title('Curva de calibración', fontsize=10)
    a2.axhline(0, color='#7f8c8d'); a2.stem(P1_P, 1000*r, basefmt=' ')
    a2.set_xlabel('P (kPa)'); a2.set_ylabel('residual (µA)'); a2.set_title('Residuales: sin patrón', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def calibracion(ruido_uA=30.0, n=7.0, desfase_mA=0.1):
    rng = np.random.default_rng(7); nn = int(n); P = np.linspace(0, 600, nn)
    I = 4 + desfase_mA + 16*P/600 + rng.normal(0, ruido_uA/1000, nn)
    m, b, R2, um, ub, r = _ajuste(P, I)
    print_es(f"m = {_c3(m*1000)} µA/kPa   |   b = {_c3(b)} mA   |   R² = {R2:.5f}   |   P para 12,0 mA = {_c3((12 - b)/m)} kPa")
    plt.figure(figsize=(8, 3.3)); plt.scatter(P, I, color='#2980b9'); plt.plot(P, m*P + b, color='#8e44ad')
    plt.xlabel('P (kPa)'); plt.ylabel('I (mA)'); plt.title('Calibración simulada', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    m, b, R2, um, ub, r = _ajuste(P1_P, P1_I); n = len(P1_P)
    print_es(f"a) n = 7 ;  ΣP = {P1_P.sum():.0f} ;  ΣI = {_c3(P1_I.sum())} ;  ΣP² = {_sci(np.sum(P1_P**2))} ;  ΣP·I = {_sci(np.sum(P1_P*P1_I))}")
    print_es(f"   m = [7·ΣPI − ΣP·ΣI]/[7·ΣP² − (ΣP)²] = {_sci(m)} mA/kPa = {_c3(m*1000)} µA/kPa")
    print_es(f"   (ideal: 16,0 mA/600 kPa = 26,7 µA/kPa)")
    print_es(f"b) b = Ī − m·P̄ = {_c3(P1_I.mean())} − {_sci(m)}·300 = {_c3(b)} mA  (el cero vivo ideal es 4,00 mA)")
    print_es(f"c) R² = {R2:.5f} ;  residuales entre {_c3(1000*r.min())} y {_c3(1000*r.max())} µA, sin patrón visible")
    print_es(f"d) Lectura de 12,0 mA:  P = (12,0 − {_c3(b)})/{_sci(m)} = {_c3((12 - b)/m)} kPa")
    print_es(f"e) Con 25,0 mA la recta daría {_c3((25 - b)/m)} kPa, fuera del rango calibrado (0–600 kPa): no es una lectura válida.")
    cf = np.polyfit(P1_P, P1_I, 1)
    print_es(f"Verificación (numpy.polyfit, método independiente): m = {_c3(cf[0]*1000)} µA/kPa ✓ , b = {_c3(cf[1])} mA ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 1 ---")
quiz_numerico("Sensibilidad del transmisor, pendiente m (en µA/kPa):",
    26.6, 0.03, "µA/kPa", "m=2,66×10⁻² mA/kPa=26,6 µA/kPa (ideal 26,7).", id_preg="AJ_p1_1", peso=2.0)
quiz_numerico("Intercepto b de la recta de calibración (en mA):",
    4.08, 0.03, "mA", "b=Ī−m·P̄=12,06−0,0266·300=4,08 mA.", id_preg="AJ_p1_2", peso=2.0)
quiz_numerico("Presión que corresponde a una lectura de 12,0 mA (en kPa):",
    298, 0.03, "kPa", "P=(12,0−4,08)/0,0266=298 kPa.", id_preg="AJ_p1_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: durante una sobrepresión el transmisor marca 25,0 mA y el operador reporta 'la presión es 787 kPa; la recta tiene R² = 0,99998, así que es confiable'. ¿Correcto o solo convincente?",
    ['Solo convincente: R² mide el ajuste dentro de 0–600 kPa; 25,0 mA está fuera del rango calibrado y la recta no puede validarlo',
     'Correcto: con un R² tan alto la recta vale para cualquier corriente que entregue el transmisor',
     'Correcto: basta con que la pendiente sea positiva para poder extrapolar',
     'Solo convincente: con 25,0 mA la presión debe ser negativa, porque supera el máximo de 20 mA'], 0,
    'El R² altísimo es real, y eso hace convincente la conclusión. Pero ningún dato respalda la recta por encima de 600 kPa: es una extrapolación. Una corriente fuera de 4–20 mA indica, sobre todo, una condición anormal que hay que revisar.',
    id_preg='AJ_p1_concep_1', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(calibracion,
    ruido_uA=FloatSlider(value=80.0,min=0,max=300,step=10,description='ruido (µA)'),
    n=widgets.IntSlider(value=4,min=3,max=15,step=1,description='n puntos'),
    desfase_mA=FloatSlider(value=0.3,min=-0.5,max=0.5,step=0.05,description='desfase (mA)'));

---
## Punto 2 — Constante de un resorte con su incertidumbre (Ing. Mecánica/Control de calidad)  ★★★  [UCM, PRC]
Un lote de resortes para válvulas debe tener **k = 250 N/m ± 1,00 %** según el fabricante. En control de calidad se cuelgan
cargas conocidas a un resorte del lote y se mide su elongación con un calibrador:

| F (N) | 10,0 | 20,0 | 30,0 | 40,0 | 50,0 | 60,0 |
|---|---|---|---|---|---|---|
| δ (mm) | 41,7 | 78,3 | 122,0 | 161,8 | 203,1 | 244,4 |

**Modelo (ley de Hooke):**  F = k·δ + b   (b debería ser compatible con cero)

**Incertidumbre estándar de la pendiente:**  u(k) = s_y/√Σ(δᵢ − δ̄)² ,  s_y = √[Σeᵢ²/(n − 2)]

> 🔧 **Aplicación en ingeniería:** en un informe de calidad, «k = 245 N/m» no basta: hay que decir cuánto confiamos en ese
> número. Con u(k) se decide si el resorte cumple la especificación o si la diferencia con el valor nominal es real y no
> producto del ruido de medición. Se ajusta F contra δ porque la carga se conoce con mucha más exactitud que la elongación;
> lo ideal en mínimos cuadrados es que el error esté en la variable del eje vertical, así que esta elección es un supuesto
> que conviene declarar.

> 🤖 **TIP-IA — Traductor de representaciones:**
> Pídele a un asistente: *"Ayúdame a conectar la gráfica de una recta de ajuste con la incertidumbre de su pendiente: quiero entender por qué u(m) depende de la dispersión de los residuales y de qué tan separados están los valores de x. Hazme dibujar dos casos y preguntarme cuál da una pendiente más confiable, sin darme fórmulas resueltas."* Primero intenta tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

try:
    PUNTO_DE_BR
except NameError:
    PUNTO_DE_BR = {}         # id_preg -> punto (botón «Mostrar solución») al que pertenece
    SOL_ANTES_BR = set()     # preguntas cuya solución se abrió antes del primer intento
    _PUNTO_ACTUAL_BR = [0]
def mostrar_solucion(func_solucion):
    # La solución es libre. Si se abre antes de intentar alguna pregunta del punto, queda anotado en la nota (no resta).
    _PUNTO_ACTUAL_BR[0] += 1; punto = _PUNTO_ACTUAL_BR[0]
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        try: pend = [k for k, p in PUNTO_DE_BR.items() if p == punto and INTENTOS_BR.get(k, [0])[0] == 0]
        except NameError: pend = []
        SOL_ANTES_BR.update(pend)
        with salida:
            salida.clear_output()
            if pend:
                print("ℹ Abriste la solución antes de intentar " + ("una pregunta" if len(pend) == 1 else "varias preguntas") + " de este punto.")
                print("  No resta puntos: solo queda anotado en tu nota, para que sepas cuáles resolviste por tu cuenta.\n")
            func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            try: vistas = sum(1 for k in PESOS_BR if k in SOL_ANTES_BR)
            except NameError: vistas = 0
            print(f"  Solución vista antes de intentar: {vistas}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if vistas: print("  ℹ En las preguntas donde abriste la solución antes, la nota no dice si sabes resolverlas sin ayuda: vuelve a intentarlas en unos días.")
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def _ajuste(x, y):
    # Mínimos cuadrados para y = m·x + b (texto guía, Ec. 3.1 y 3.2) con incertidumbres estándar de m y b
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    m = (n*np.sum(x*y) - x.sum()*y.sum())/(n*np.sum(x*x) - x.sum()**2); b = y.mean() - m*x.mean()
    r = y - (m*x + b); R2 = 1 - np.sum(r*r)/np.sum((y - y.mean())**2)
    sy = np.sqrt(np.sum(r*r)/(n - 2)); Sxx = np.sum((x - x.mean())**2)
    return m, b, R2, sy/np.sqrt(Sxx), sy*np.sqrt(np.sum(x*x)/(n*Sxx)), r

P2_F = np.array([10, 20, 30, 40, 50, 60.]); P2_d = np.array([41.7, 78.3, 122.0, 161.8, 203.1, 244.4])/1000

def esquema():
    k, b, R2, uk, ub, r = _ajuste(P2_d, P2_F)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.scatter(1000*P2_d, P2_F, color='#2980b9', zorder=5); dd = np.linspace(0, 0.26, 50)
    a1.plot(1000*dd, k*dd + b, color='#8e44ad'); a1.plot(1000*dd, (k + 2*uk)*dd + b, ':', color='#7f8c8d'); a1.plot(1000*dd, (k - 2*uk)*dd + b, ':', color='#7f8c8d')
    a1.set_xlabel('elongación δ (mm)'); a1.set_ylabel('carga F (N)'); a1.set_title('Ajuste y rectas con k ± 2u(k)', fontsize=10)
    for kk, et, col in ((250*0.99, 'límite −1 %', '#c0392b'), (250*1.01, 'límite +1 %', '#c0392b')):
        a2.axvline(kk, color=col, ls='--')
    xs = np.linspace(235, 260, 400); a2.plot(xs, np.exp(-0.5*((xs - k)/uk)**2), color='#8e44ad', lw=2)
    a2.axvspan(247.5, 252.5, color='#27ae60', alpha=0.12); a2.text(247.8, 1.02, 'especificación', fontsize=8, color='#27ae60')
    a2.set_xlabel('k (N/m)'); a2.set_yticks([]); a2.set_ylim(0, 1.12); a2.set_title('Estimación de k frente a la especificación', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def incertidumbre(ruido_mm=1.5, n=6.0, rango_N=50.0):
    rng = np.random.default_rng(11); nn = int(n); F = np.linspace(10, 10 + rango_N, nn); d = F/246 + rng.normal(0, ruido_mm/1000, nn)
    k, b, R2, uk, ub, r = _ajuste(d, F)
    print_es(f"k = {_c3(k)} N/m   |   u(k) = {_c3(uk)} N/m ({_c3(100*uk/k)} %)   |   R² = {R2:.5f}")
    plt.figure(figsize=(8, 3.3)); plt.scatter(1000*d, F, color='#2980b9'); plt.plot(1000*d, k*d + b, color='#8e44ad')
    plt.xlabel('δ (mm)'); plt.ylabel('F (N)'); plt.title('Ensayo simulado', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    k, b, R2, uk, ub, r = _ajuste(P2_d, P2_F); n = len(P2_F)
    print_es(f"a) Ajuste F = k·δ + b con δ en metros:  k = {_c3(k)} N/m ;  b = {_c3(b)} N ;  R² = {R2:.5f}")
    sy = np.sqrt(np.sum(r*r)/(n - 2)); Sxx = np.sum((P2_d - P2_d.mean())**2)
    print_es(f"b) s_y = √[Σe²/(6 − 2)] = {_c3(sy)} N ;  Σ(δ − δ̄)² = {_sci(Sxx)} m² ;  u(k) = {_c3(sy)}/√{_sci(Sxx)} = {_c3(uk)} N/m")
    print_es(f"   Reporte: k = ({_c3(k)} ± {_c3(uk)}) N/m  (incertidumbre estándar, {_c3(100*uk/k)} %)")
    print_es(f"c) b = ({_c3(b)} ± {_c3(ub)}) N: |b| < u(b), compatible con cero, como predice la ley de Hooke.")
    print_es(f"d) Especificación: 247,5 a 252,5 N/m. El valor medido queda {_c3(247.5 - k)} N/m por debajo del límite inferior,")
    print_es(f"   y está a {_c3((250 - k)/uk)}·u(k) del nominal: la diferencia es mayor que el ruido de medición; el resorte no cumple.")
    from scipy.stats import linregress
    L = linregress(P2_d, P2_F)
    print_es(f"Verificación (scipy.stats.linregress, independiente): k = {_c3(L.slope)} N/m ✓ , u(k) = {_c3(L.stderr)} N/m ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 2 ---")
quiz_numerico("Constante k del resorte por mínimos cuadrados (en N/m):",
    245, 0.02, "N/m", "Pendiente de F contra δ (en m): k=245 N/m.", id_preg="AJ_p2_1", peso=3.0)
quiz_numerico("Incertidumbre estándar de la pendiente, u(k) (en N/m):",
    2.24, 0.05, "N/m", "s_y=0,383 N; Σ(δ−δ̄)²=2,91×10⁻² m²; u(k)=0,383/√(2,91×10⁻²)=2,24 N/m.", id_preg="AJ_p2_2", peso=3.0)
quiz_opcion_multiple("Con k = (245 ± 2,24) N/m y la especificación 250 N/m ± 1,00 %, ¿qué se concluye?",
    ['No cumple: queda por debajo de 247,5 N/m y la diferencia con el nominal es más de dos veces u(k)',
     'Cumple: la diferencia de 5 N/m está dentro del ± 1,00 % porque 1 % de 250 son 2,5 N/m por lado',
     'Cumple: si se suma la incertidumbre, 245 + 2,24 = 247 N/m, que está dentro de la especificación',
     'No se puede concluir nada: la incertidumbre siempre impide decidir'], 0,
    "El intervalo de especificación es 247,5–252,5 N/m. La distancia al nominal (5 N/m) es 2,2·u(k): no se explica por el ruido. Además, 245 + 2,24 = 247,2 N/m sigue por fuera.",
    id_preg="AJ_p2_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un practicante escribe 'R² = 0,9997, así que la incertidumbre de k es despreciable y podemos reportar k = 245,068 N/m'. ¿Correcto o solo convincente?",
    ['Solo convincente: R² no es la incertidumbre; u(k) = 2,24 N/m (0,915 %) es comparable a la tolerancia de ± 1,00 % y no se puede omitir',
     'Correcto: un R² mayor que 0,999 implica que todas las cifras de la pendiente son significativas',
     'Correcto: la incertidumbre de la pendiente solo importa si R² es menor que 0,95',
     'Solo convincente: con R² = 0,9997 la incertidumbre es exactamente 0,03 % de k'], 0,
    'Un R² alto dice que la recta explica casi toda la variación, y eso lo hace convincente. Pero la confianza en la pendiente depende de los residuales y del rango de δ. Aquí u(k) es casi del tamaño de la tolerancia: reportar seis cifras es falsa precisión.',
    id_preg='AJ_p2_concep_2', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(incertidumbre),
    ruido_mm=FloatSlider(value=3.0,min=0.1,max=6,step=0.1,description='ruido δ (mm)'),
    n=widgets.IntSlider(value=4,min=3,max=20,step=1,description='n puntos'),
    rango_N=FloatSlider(value=20.0,min=5,max=100,step=5,description='rango F (N)'));

---
## Punto 3 — Vertedero triangular: descubrir una ley potencial (Ing. Civil/Hidráulica)  ★★★  [PRC, ULC]
En un canal de laboratorio se mide el caudal Q que pasa por un vertedero triangular de 90° para distintas cargas H
(altura del agua sobre el vértice):

| H (m) | 0,050 | 0,080 | 0,110 | 0,140 | 0,170 | 0,200 |
|---|---|---|---|---|---|---|
| Q (L/s) | 0,774 | 2,475 | 5,586 | 10,049 | 16,657 | 24,591 |

**Modelo candidato:**  Q = C·Hⁿ  (ley potencial)

**Linealización:**  log Q = log C + n·log H

**Pendiente en log–log = n ;  intercepto = log C**  (con Q en m³/s y H en m)

**Valor teórico:** el análisis de energía da n = 5/2 para un vertedero triangular; para 90°, C ≈ 1,38 m^(1/2)/s es un
valor típico de la literatura (supone un coeficiente de descarga cercano a 0,59; no verificado aquí con norma).

> 🔧 **Aplicación en ingeniería:** los vertederos son la forma más simple de medir caudal en canales, plantas de
> tratamiento y distritos de riego: basta medir una altura. Pero la relación no es lineal, y calibrarla con una recta
> en Q contra H produciría errores sistemáticos en los extremos del rango, justo donde más importan.

> 🤖 **TIP-IA — Abogado del diablo:**
> Dile a un asistente: *"Voy a sostener que si una recta en Q contra H da R² = 0,94, el modelo lineal es suficientemente bueno para un vertedero. Dame el mejor contraargumento usando la gráfica de residuales y pregúntame qué patrón esperaría ver si el modelo fuera el correcto. No me des números."* Primero razona tú.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

try:
    PUNTO_DE_BR
except NameError:
    PUNTO_DE_BR = {}         # id_preg -> punto (botón «Mostrar solución») al que pertenece
    SOL_ANTES_BR = set()     # preguntas cuya solución se abrió antes del primer intento
    _PUNTO_ACTUAL_BR = [0]
def mostrar_solucion(func_solucion):
    # La solución es libre. Si se abre antes de intentar alguna pregunta del punto, queda anotado en la nota (no resta).
    _PUNTO_ACTUAL_BR[0] += 1; punto = _PUNTO_ACTUAL_BR[0]
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        try: pend = [k for k, p in PUNTO_DE_BR.items() if p == punto and INTENTOS_BR.get(k, [0])[0] == 0]
        except NameError: pend = []
        SOL_ANTES_BR.update(pend)
        with salida:
            salida.clear_output()
            if pend:
                print("ℹ Abriste la solución antes de intentar " + ("una pregunta" if len(pend) == 1 else "varias preguntas") + " de este punto.")
                print("  No resta puntos: solo queda anotado en tu nota, para que sepas cuáles resolviste por tu cuenta.\n")
            func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            try: vistas = sum(1 for k in PESOS_BR if k in SOL_ANTES_BR)
            except NameError: vistas = 0
            print(f"  Solución vista antes de intentar: {vistas}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if vistas: print("  ℹ En las preguntas donde abriste la solución antes, la nota no dice si sabes resolverlas sin ayuda: vuelve a intentarlas en unos días.")
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def _ajuste(x, y):
    # Mínimos cuadrados para y = m·x + b (texto guía, Ec. 3.1 y 3.2) con incertidumbres estándar de m y b
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    m = (n*np.sum(x*y) - x.sum()*y.sum())/(n*np.sum(x*x) - x.sum()**2); b = y.mean() - m*x.mean()
    r = y - (m*x + b); R2 = 1 - np.sum(r*r)/np.sum((y - y.mean())**2)
    sy = np.sqrt(np.sum(r*r)/(n - 2)); Sxx = np.sum((x - x.mean())**2)
    return m, b, R2, sy/np.sqrt(Sxx), sy*np.sqrt(np.sum(x*x)/(n*Sxx)), r

P3_H = np.array([0.050, 0.080, 0.110, 0.140, 0.170, 0.200]); P3_Q = np.array([0.774, 2.475, 5.586, 10.049, 16.657, 24.591])/1000

def esquema():
    import matplotlib.ticker as _tk
    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(11, 3.6))
    m, b, R2, um, ub, r = _ajuste(P3_H, P3_Q)
    a1.scatter(P3_H, 1000*P3_Q, color='#2980b9', zorder=5); a1.plot(P3_H, 1000*(m*P3_H + b), color='#c0392b', label='recta en Q–H')
    a1.set_xlabel('H (m)'); a1.set_ylabel('Q (L/s)'); a1.legend(fontsize=8); a1.set_title('Escala lineal', fontsize=10)
    a2.axhline(0, color='#7f8c8d'); a2.stem(P3_H, 1000*r, basefmt=' ', linefmt='#c0392b', markerfmt='o')
    a2.set_xlabel('H (m)'); a2.set_ylabel('residual (L/s)'); a2.set_title('Residuales de la recta: forma de U', fontsize=10)
    n, lc, R2l, un, ulc, rl = _ajuste(np.log10(P3_H), np.log10(P3_Q))
    a3.loglog(P3_H, 1000*P3_Q, 'o', color='#2980b9'); hh = np.linspace(0.045, 0.21, 50); a3.loglog(hh, 1000*10**lc*hh**n, color='#27ae60')
    for ax in (a3.xaxis, a3.yaxis):
        ax.set_major_formatter(_tk.FuncFormatter(lambda v, _: f'{v:g}'.replace('.', ','))); ax.set_minor_formatter(_tk.NullFormatter())
    a3.set_xticks([0.05, 0.1, 0.2]); a3.set_yticks([1, 2, 5, 10, 20])
    a3.set_xlabel('H (m, log)'); a3.set_ylabel('Q (L/s, log)'); a3.set_title('Escala log–log: una recta', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def vertedero(n_real=2.5, ruido_pct=1.0):
    rng = np.random.default_rng(5); H = np.linspace(0.05, 0.2, 6); Q = 1.38*H**n_real*(1 + rng.normal(0, ruido_pct/100, 6))
    n, lc, R2l, un, ulc, rl = _ajuste(np.log10(H), np.log10(Q)); m, b, R2, um, ub, r = _ajuste(H, Q)
    print_es(f"log–log: n = {_c3(n)} ± {_c3(un)} ; C = {_c3(10**lc)}   |   recta en Q–H: R² = {R2:.4f}")
    plt.figure(figsize=(8, 3.3)); plt.axhline(0, color='#7f8c8d'); plt.plot(H, 1000*r, 'o-', color='#c0392b', label='residuales de la recta')
    plt.plot(H, 1000*(Q - 10**lc*H**n), 's-', color='#27ae60', label='residuales de la ley potencial'); plt.legend(fontsize=8)
    plt.xlabel('H (m)'); plt.ylabel('residual (L/s)'); plt.title('¿Qué modelo deja residuales sin patrón?', fontsize=10); plt.tight_layout(); plt.show()

def solucion():
    X = np.log10(P3_H); Y = np.log10(P3_Q)
    n, lc, R2l, un, ulc, rl = _ajuste(X, Y); m, b, R2, um, ub, r = _ajuste(P3_H, P3_Q)
    print_es(f"a) Recta directa Q = m·H + b:  R² = {R2:.4f} ;  residuales (L/s): " + ' ; '.join(_c3(1000*v) for v in r))
    print_es("   Signos + − − − − +: forma de U. El modelo lineal es sistemáticamente incorrecto, aunque R² parezca alto.")
    print_es(f"b) log–log:  n = {_c3(n)} ± {_c3(un)} ;  log C = {_c3(lc)} → C = 10^{_c3(lc)} = {_c3(10**lc)} m^(1/2)/s ;  R² = {R2l:.5f}")
    print_es(f"c) n es compatible con 5/2 = 2,50: la diferencia es {_c3(abs(n - 2.5)/un)} veces su incertidumbre.")
    print_es(f"d) Caudal para H = 0,150 m:  Q = {_c3(10**lc)}·0,150^{_c3(n)} = {_c3(1000*10**lc*0.15**n)} L/s  (la recta daría {_c3(1000*(m*0.15 + b))} L/s)")
    from scipy.optimize import curve_fit
    p, _ = curve_fit(lambda h, C, nn: C*h**nn, P3_H, P3_Q, p0=[1.0, 2.0])
    print_es(f"Verificación (ajuste no lineal directo con scipy.optimize.curve_fit): C = {_c3(p[0])} , n = {_c3(p[1])}")
    print_es(f"   Coinciden dentro de {_c3(100*abs(p[0] - 10**lc)/(10**lc))} % en C y {_c3(100*abs(p[1] - n)/n)} % en n: el ajuste en logaritmos pondera los errores")
    print_es("   relativos y el ajuste directo los absolutos (dominan los caudales grandes). Ambos confirman n ≈ 5/2.")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 3 ---")
quiz_numerico("Exponente n obtenido del ajuste log–log (adimensional):",
    2.50, 0.02, "", "Pendiente de log Q contra log H: n=2,50 ± 0,00865.", id_preg="AJ_p3_1", peso=3.0)
quiz_numerico("Coeficiente C del vertedero, con Q en m³/s y H en m:",
    1.39, 0.03, "", "log C=0,142 → C=10^0,142=1,39 (valor típico ≈1,38).", id_preg="AJ_p3_2", peso=3.0)
quiz_numerico("Caudal para H = 0,150 m con la ley potencial ajustada (en L/s):",
    12.0, 0.03, "L/s", "Q=1,39·0,150^2,50=12,0 L/s (la recta daría 14,0 L/s).", id_preg="AJ_p3_3", peso=2.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un técnico dice 'la recta en Q contra H tiene R² = 0,942; eso es un buen ajuste, así que usamos la recta y nos ahorramos los logaritmos'. ¿Correcto o solo convincente?",
    ['Solo convincente: los residuales de la recta forman una U (+ − − − − +); el modelo es sistemáticamente incorrecto aunque R² sea alto',
     'Correcto: cualquier R² mayor que 0,9 confirma que el modelo lineal es adecuado',
     'Correcto: los logaritmos solo se usan cuando R² es menor que 0,5',
     'Solo convincente: con R² = 0,942 el ajuste es malo porque R² debería ser exactamente 1'], 0,
    'R² = 0,942 suena alto, y eso lo hace convincente (el texto guía, sección 3.6, advierte esta trampa). El criterio decisivo es la gráfica de residuales: un patrón sistemático indica que el modelo no es el correcto. La última opción exige un R² perfecto, que ningún dato real da.',
    id_preg='AJ_p3_concep_3', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(vertedero),
    n_real=FloatSlider(value=1.5,min=1.0,max=3.0,step=0.1,description='n verdadero'),
    ruido_pct=FloatSlider(value=4.0,min=0,max=10,step=0.5,description='ruido (%)'));

---
## Punto 4 — Enfriamiento del aceite de un transformador (Ing. Eléctrica/Mantenimiento)  ★★★  [PRC, ULC]
Después de desconectar un transformador de distribución, se registra la temperatura de su aceite. La temperatura
ambiente es **T_amb = 18,0 °C** (datos didácticos):

| t (h) | 0 | 0,50 | 1,00 | 1,50 | 2,00 | 3,00 | 4,00 | 5,00 |
|---|---|---|---|---|---|---|---|---|
| T (°C) | 80,0 | 68,2 | 59,3 | 52,4 | 45,9 | 36,3 | 30,4 | 26,4 |

**Modelo (enfriamiento de Newton):**  T(t) = T_amb + ΔT₀·e^(−t/τ)

**Linealización:**  ln(T − T_amb) = ln ΔT₀ − t/τ   → pendiente = −1/τ ,  intercepto = ln ΔT₀

**Tiempo para llegar a una temperatura T*:**  t* = τ·ln[ΔT₀/(T* − T_amb)]

> 🔧 **Aplicación en ingeniería:** la constante de tiempo τ dice qué tan rápido se calienta o se enfría el transformador;
> se usa para decidir cuánto tiempo puede trabajar con sobrecarga o cuándo es seguro intervenirlo. Linealizar bien exige
> restar la asíntota correcta: el aceite no tiende a 0 °C, sino a la temperatura ambiente.

> 🤖 **TIP-IA — Generador de problemas:**
> Pídele a un asistente: *"Créame una variante con otro objeto que se enfría o se calienta (otra temperatura ambiente, otra constante de tiempo) y una tabla de 6 a 8 mediciones con ruido. Pídeme linealizar, obtener τ y predecir el tiempo para llegar a una temperatura dada. No me des la solución hasta que yo la intente."* Primero resuelves tú, después verificas.


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

try:
    PUNTO_DE_BR
except NameError:
    PUNTO_DE_BR = {}         # id_preg -> punto (botón «Mostrar solución») al que pertenece
    SOL_ANTES_BR = set()     # preguntas cuya solución se abrió antes del primer intento
    _PUNTO_ACTUAL_BR = [0]
def mostrar_solucion(func_solucion):
    # La solución es libre. Si se abre antes de intentar alguna pregunta del punto, queda anotado en la nota (no resta).
    _PUNTO_ACTUAL_BR[0] += 1; punto = _PUNTO_ACTUAL_BR[0]
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        try: pend = [k for k, p in PUNTO_DE_BR.items() if p == punto and INTENTOS_BR.get(k, [0])[0] == 0]
        except NameError: pend = []
        SOL_ANTES_BR.update(pend)
        with salida:
            salida.clear_output()
            if pend:
                print("ℹ Abriste la solución antes de intentar " + ("una pregunta" if len(pend) == 1 else "varias preguntas") + " de este punto.")
                print("  No resta puntos: solo queda anotado en tu nota, para que sepas cuáles resolviste por tu cuenta.\n")
            func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            try: vistas = sum(1 for k in PESOS_BR if k in SOL_ANTES_BR)
            except NameError: vistas = 0
            print(f"  Solución vista antes de intentar: {vistas}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if vistas: print("  ℹ En las preguntas donde abriste la solución antes, la nota no dice si sabes resolverlas sin ayuda: vuelve a intentarlas en unos días.")
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


try:
    import matplotlib.pyplot as _plt_style
    _plt_style.rcParams['axes.spines.top']=False
    _plt_style.rcParams['axes.spines.right']=False
    _plt_style.rcParams['axes.titlesize']=12
    _plt_style.rcParams['axes.titleweight']='bold'
    _plt_style.rcParams['axes.titlecolor']='#2c3e50'
    _plt_style.rcParams['axes.labelsize']=10
    _plt_style.rcParams['axes.labelcolor']='#34495e'
    _plt_style.rcParams['font.size']=10
    _plt_style.rcParams['axes.grid']=True
    _plt_style.rcParams['grid.alpha']=0.25
    _plt_style.rcParams['grid.linestyle']='--'
    _plt_style.rcParams['axes.edgecolor']='#7f8c8d'
    _plt_style.rcParams['xtick.color']='#34495e'
    _plt_style.rcParams['ytick.color']='#34495e'
    _plt_style.rcParams['figure.facecolor']='white'
    _plt_style.rcParams['axes.facecolor']='#fcfcfd'
except Exception:
    pass
# --- fin estilo ---


def _c3(x):
    # 3 cifras significativas conservando ceros finales (8,00 y no 8); notación ×10ⁿ desde 1000
    if x != 0 and abs(x) >= 1000: return _sci(x)
    t = f"{x:#.3g}"
    return t[:-1] if t.endswith('.') else t

def _sci(x):
    e = int(np.floor(np.log10(abs(x)))); m = x/10**e
    if round(m, 2) >= 10: m /= 10; e += 1
    return f"{m:.2f}×10" + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))

def _3c(x):
    # 3 cifras significativas con coma decimal; ×10ⁿ fuera de 0,001 ≤ |x| < 1000 (textos de gráficas)
    import math
    try: x = float(x)
    except (TypeError, ValueError): return str(x)
    if x == 0 or not math.isfinite(x): return str(x).replace('.', ',') if x else '0'
    x = float(f"{x:.2e}")
    if abs(x) >= 1000 or abs(x) < 0.001:
        e = int(math.floor(math.log10(abs(x)))); m = x / 10**e
        if round(abs(m), 2) >= 10: m /= 10; e += 1
        return f"{m:.2f}".replace('.', ',').replace('-', '−') + '×10' + str(e).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
    t = f"{x:#.3g}"
    return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')

def _ajuste(x, y):
    # Mínimos cuadrados para y = m·x + b (texto guía, Ec. 3.1 y 3.2) con incertidumbres estándar de m y b
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    m = (n*np.sum(x*y) - x.sum()*y.sum())/(n*np.sum(x*x) - x.sum()**2); b = y.mean() - m*x.mean()
    r = y - (m*x + b); R2 = 1 - np.sum(r*r)/np.sum((y - y.mean())**2)
    sy = np.sqrt(np.sum(r*r)/(n - 2)); Sxx = np.sum((x - x.mean())**2)
    return m, b, R2, sy/np.sqrt(Sxx), sy*np.sqrt(np.sum(x*x)/(n*Sxx)), r

P4_t = np.array([0, 0.5, 1, 1.5, 2, 3, 4, 5.]); P4_T = np.array([80.0, 68.2, 59.3, 52.4, 45.9, 36.3, 30.4, 26.4]); P4_Tamb = 18.0

def esquema():
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    a1.scatter(P4_t, P4_T, color='#2980b9', zorder=5); a1.axhline(P4_Tamb, ls='--', color='#7f8c8d'); a1.text(3.2, 19.5, 'T_amb', fontsize=8, color='#7f8c8d')
    a1.set_xlabel('t (h)'); a1.set_ylabel('T (°C)'); a1.set_title('Datos: decaimiento hacia T_amb', fontsize=10)
    a2.scatter(P4_t, np.log(P4_T - P4_Tamb), color='#27ae60', label='ln(T − T_amb)'); a2.scatter(P4_t, np.log(P4_T), color='#c0392b', marker='s', label='ln T (sin restar T_amb)')
    m1, b1, *_ = _ajuste(P4_t, np.log(P4_T - P4_Tamb)); m2, b2, *_ = _ajuste(P4_t, np.log(P4_T))
    a2.plot(P4_t, m1*P4_t + b1, color='#27ae60'); a2.plot(P4_t, m2*P4_t + b2, color='#c0392b', ls=':')
    a2.set_xlabel('t (h)'); a2.set_ylabel('logaritmo natural'); a2.legend(fontsize=8); a2.set_title('Linealizar: ¿qué se resta?', fontsize=10)
    plt.tight_layout(); plt.show()
esquema()

def enfriamiento(Tamb_supuesta=18.0, T_obj=30.0):
    m, b, R2, um, ub, r = _ajuste(P4_t, np.log(np.clip(P4_T - Tamb_supuesta, 1e-9, None)))
    if np.any(P4_T - Tamb_supuesta <= 0):
        print_es("⛔ BLOQUEADO: con esa temperatura ambiente supuesta algún dato queda por debajo de ella (ln de un número ≤ 0)."); return
    tau = -1/m; dT0 = np.exp(b)
    print_es(f"τ = {_c3(tau)} h   |   ΔT₀ = {_c3(dT0)} °C   |   R² = {R2:.5f}")
    if T_obj <= Tamb_supuesta: print_es("⛔ BLOQUEADO: el aceite nunca baja de la temperatura ambiente supuesta.")
    else: print_es(f"Tiempo para llegar a T = {_c3(T_obj)} °C: {_c3(tau*np.log(dT0/(T_obj - Tamb_supuesta)))} h")
    tt = np.linspace(0, 8, 200)
    plt.figure(figsize=(8, 3.3)); plt.scatter(P4_t, P4_T, color='#2980b9'); plt.plot(tt, Tamb_supuesta + dT0*np.exp(-tt/tau), color='#8e44ad')
    plt.axhline(T_obj, ls=':', color='#c0392b'); plt.xlabel('t (h)'); plt.ylabel('T (°C)'); plt.title('Modelo ajustado y extrapolación', fontsize=10)
    plt.tight_layout(); plt.show()

def solucion():
    Y = np.log(P4_T - P4_Tamb); m, b, R2, um, ub, r = _ajuste(P4_t, Y); tau = -1/m; dT0 = np.exp(b)
    print_es("a) Se calcula y = ln(T − 18,0) para cada dato: " + ' ; '.join(_c3(v) for v in Y))
    print_es(f"b) Recta de y contra t:  pendiente = {_c3(m)} h⁻¹ → τ = −1/pendiente = {_c3(tau)} h ;  ΔT₀ = e^{_c3(b)} = {_c3(dT0)} °C ;  R² = {R2:.5f}")
    print_es(f"c) Para T* = 30,0 °C:  t* = {_c3(tau)}·ln({_c3(dT0)}/12,0) = {_c3(tau*np.log(dT0/12))} h")
    m2, b2, R22, *_ = _ajuste(P4_t, np.log(P4_T))
    print_es(f"d) Sin restar T_amb: τ aparente = {_c3(-1/m2)} h (R² = {R22:.4f}): casi el doble, y los puntos se curvan en ln T.")
    from scipy.optimize import curve_fit
    p, _ = curve_fit(lambda t, a, c, d: a + c*np.exp(-t/d), P4_t, P4_T, p0=[20, 60, 2])
    print_es(f"Verificación (ajuste no lineal con T_amb también libre, curve_fit): τ = {_c3(p[2])} h ✓ , T_amb = {_c3(p[0])} °C (≈ 18,0) ✓")
mostrar_solucion(solucion)

print("\n--- Cuestionario Punto 4 ---")
quiz_numerico("Constante de tiempo τ del enfriamiento (en h):",
    2.50, 0.03, "h", "Pendiente de ln(T−18,0) contra t: −0,401 h⁻¹ → τ=2,50 h.", id_preg="AJ_p4_1", peso=3.0)
quiz_numerico("Diferencia inicial ΔT₀ según el ajuste (en °C):",
    61.8, 0.03, "°C", "Intercepto ln ΔT₀=4,12 → ΔT₀=61,8 °C.", id_preg="AJ_p4_2", peso=2.0)
quiz_numerico("Tiempo desde la desconexión para que el aceite llegue a 30,0 °C (en h):",
    4.09, 0.03, "h", "t*=2,50·ln(61,8/12,0)=2,50·1,64=4,09 h.", id_preg="AJ_p4_3", peso=3.0)
quiz_opcion_multiple("DEFIENDE TU CONCLUSIÓN: un estudiante linealiza con ln T (sin restar la temperatura ambiente), obtiene τ = 4,49 h con R² = 0,986 y dice 'R² alto, así que τ = 4,49 h es confiable'. ¿Correcto o solo convincente?",
    ['Solo convincente: el modelo correcto decae hacia T_amb, no hacia 0 °C; sin restarla los puntos se curvan y τ sale casi el doble',
     'Correcto: restar o no la temperatura ambiente solo cambia el intercepto, no la pendiente',
     'Correcto: con R² mayor que 0,98 cualquier linealización es válida',
     'Solo convincente: lo correcto es usar log₁₀ en vez de ln; con eso τ queda bien'], 0,
    'R² = 0,986 parece alto, y eso lo hace convincente. Pero el modelo físico decae hacia 18,0 °C: ln T no es lineal en t y la recta forzada da τ = 4,49 h en vez de 2,50 h. Cambiar ln por log₁₀ solo reescala la pendiente; no corrige el modelo.',
    id_preg='AJ_p4_concep_4', peso=2.0)

def _interact_coma(fn, _crudos=(), **controles):
    # Como interact, con lectura de sliders en coma decimal. La salida NO se captura con «with salida:»:
    # se calculan el texto y las figuras y se escriben con append_stdout/append_display_data
    # (evita bloqueos de Voilà en Binder y funciona también en JupyterLite).
    import ipywidgets as _w, io as _io, contextlib as _cl
    import matplotlib.pyplot as _plt
    def _fmt(v, paso, crudo=False):
        # Lectura con coma decimal y 3 cifras significativas (×10ⁿ fuera de 0,001 ≤ |v| < 1000).
        # IntSlider: entero exacto. «crudo»: decimales del paso (slider que representa una medición sin redondear).
        import math
        if isinstance(v, int) and not isinstance(v, bool): return str(v)
        if crudo:
            s = repr(float(paso)); dec = 0 if 'e' in s else len(s.split('.')[1].rstrip('0'))
            return f"{v:.{min(dec, 4)}f}".replace('.', ',').replace('-', '−')
        v = float(v)
        if v == 0 or not math.isfinite(v): return '0' if v == 0 else str(v)
        v = float(f"{v:.2e}")
        if abs(v) >= 1000 or abs(v) < 0.001:
            m, e = f"{v:.2e}".split('e')
            return m.replace('.', ',').replace('-', '−') + '×10' + str(int(e)).translate(str.maketrans('-0123456789', '⁻⁰¹²³⁴⁵⁶⁷⁸⁹'))
        t = f"{v:#.3g}"
        return (t[:-1] if t.endswith('.') else t).replace('.', ',').replace('-', '−')
    filas = []
    for nombre, c in controles.items():
        if isinstance(c, (_w.FloatSlider, _w.IntSlider)):
            c.readout = False
            lab = _w.Label(layout=_w.Layout(min_width='90px'))
            def _act(*_, c=c, lab=lab, cr=(nombre in _crudos)): lab.value = _fmt(c.value, c.step, cr)
            c.observe(_act, 'value'); _act()
            filas.append(_w.HBox([c, lab]))
        else:
            filas.append(c)
    salida = _w.Output()
    def _ejecutar(*_):
        partes, buf = [], _io.StringIO()
        def _vaciar_texto():
            t = buf.getvalue()
            if t: partes.append(('t', t)); buf.seek(0); buf.truncate(0)
        def _capturar(*a, **k):
            _vaciar_texto()
            for num in _plt.get_fignums():
                b = _io.BytesIO(); _plt.figure(num).savefig(b, format='png', bbox_inches='tight')
                partes.append(('i', b.getvalue()))
            _plt.close('all')
        _show = _plt.show; _plt.show = _capturar
        try:
            with _cl.redirect_stdout(buf):
                try: fn(**{k: c.value for k, c in controles.items()})
                except Exception as e: print('⚠ No se pudo calcular con estos valores:', e)
            _capturar()
        finally:
            _plt.show = _show
        _vaciar_texto()
        # Reemplazo en UNA sola asignación: la salida nunca queda vacía (sin saltos de la página)
        import base64 as _b64
        nuevas = []
        for tipo, val in partes:
            if tipo == 't': nuevas.append({'output_type': 'stream', 'name': 'stdout', 'text': val})
            else: nuevas.append({'output_type': 'display_data', 'metadata': {},
                                 'data': {'image/png': _b64.b64encode(val).decode('ascii'), 'text/plain': '<Figure>'}})
        salida.outputs = tuple(nuevas)
    for c in controles.values():
        c.observe(_ejecutar, 'value')
    _ejecutar()
    display(_w.VBox(filas + [salida]))

def _explora_sin_cifras(fn):
    # Opción 3: exploración cualitativa; las cifras de este punto solo están en «Mostrar solución».
    import functools, io, contextlib, re as _re
    _pat = _re.compile(r'(?<![\w.,/])[-−]?\d+(?:[.,]\d+)?(?:\s?×\s?10[⁻⁰¹²³⁴⁵⁶⁷⁸⁹]+|[eE][+-]?\d+)?(?![\wπ·(]|[.,]\d)')
    def _m(t): return _pat.sub('…', t)
    def _ocultar():
        for num in plt.get_fignums():
            fig = plt.figure(num)
            textos = list(fig.texts) + ([fig._suptitle] if getattr(fig, '_suptitle', None) else [])
            for ax in fig.axes:
                textos += [ax.title, ax._left_title, ax._right_title] + list(ax.texts)
                if ax.get_legend(): textos += list(ax.get_legend().get_texts())
            for t in textos: t.set_text(_m(t.get_text()))
    @functools.wraps(fn)
    def envuelta(*a, **k):
        _show = plt.show
        def _show2(*sa, **sk):
            _ocultar(); _show(*sa, **sk)
        buf = io.StringIO(); plt.show = _show2
        try:
            with contextlib.redirect_stdout(buf): fn(*a, **k)
            _ocultar()
        finally:
            plt.show = _show
        for linea in buf.getvalue().splitlines():
            print(linea if ('⚠' in linea or '⛔' in linea) else _m(linea))
        print("(Exploración cualitativa: observa tendencias. Las cifras de este punto están en «Mostrar solución».)")
    return envuelta

print("\n--- 🔬 Explora después de responder ---")
print("Los controles NO arrancan en los datos del enunciado. Primero resuelve y responde;\nluego explora tendencias, casos límite y comprueba tu razonamiento.")
_interact_coma(_explora_sin_cifras(enfriamiento),
    Tamb_supuesta=FloatSlider(value=10.0,min=0,max=26,step=0.5,description='T_amb supuesta (°C)'),
    T_obj=FloatSlider(value=25.0,min=19,max=60,step=0.5,description='T objetivo (°C)'));

---
## ✅ Fin del taller complementario de Ajuste de Datos y Regresión (Física BR)
Calibrar, reportar con incertidumbre, linealizar y revisar residuales: el mismo método sirve para un sensor, un resorte,
un vertedero o un transformador. Experimenta con los sliders y **predice** antes de pulsar "Mostrar solución".
*Física interactiva para ingeniería — 2026-2.*


In [ ]:
# --- setup autónomo (no requiere ejecutar otras celdas) ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as _tk_br
if not getattr(_tk_br.ScalarFormatter, '_coma_br', False):
    # coma decimal en las marcas de los ejes y en el texto de desplazamiento (offset)
    _fmt_br_call, _fmt_br_off = _tk_br.ScalarFormatter.__call__, _tk_br.ScalarFormatter.get_offset
    _tk_br.ScalarFormatter.__call__ = lambda self, x, pos=None: _fmt_br_call(self, x, pos).replace('.', ',')
    def _off_br(self):
        import re as _re_br
        sup = str.maketrans('−-+0123456789', '⁻⁻⁺⁰¹²³⁴⁵⁶⁷⁸⁹')
        t = _fmt_br_off(self).replace('.', ',')
        t = _re_br.sub(r'^1e([−+-]?\d+)', lambda m: '×10' + m.group(1).translate(sup), t)
        return _re_br.sub(r'(\d)e([−+-]?\d+)', lambda m: m.group(1) + '×10' + m.group(2).translate(sup), t)
    _tk_br.ScalarFormatter.get_offset = _off_br
    _tk_br.ScalarFormatter._coma_br = True
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, Button, Text, VBox, Output, HTML
from IPython.display import display
# --- pantalla angosta (celular) ---
try:
    from IPython.display import HTML as _HTML_br
    display(_HTML_br('<style>'
        '.widget-radio{width:100% !important;max-width:100%;height:auto !important}'
        '.widget-radio-box{width:100%;max-width:100%}'
        '.widget-radio-box label{display:flex !important;flex-direction:row-reverse;justify-content:flex-end;align-items:flex-start;gap:8px;'
        'white-space:normal !important;height:auto !important;line-height:1.35 !important;margin:3px 0 7px 0;overflow-wrap:anywhere}'
        '.widget-radio-box input{flex:0 0 auto;margin:3px 0 0 0}'
        '.jp-RenderedText pre,.widget-output pre{word-break:normal !important;overflow-wrap:anywhere;white-space:pre-wrap !important}'
        '.widget-html-content{white-space:normal;line-height:1.4}'
        '.widget-text{max-width:100%}'
        '</style>'))
except Exception:
    pass
if not getattr(plt, '_dos_paneles_br', False):
    def _show_br(*a, _orig=plt.show, **k):
        # Figura de dos paneles lado a lado: se entrega como dos imágenes que se apilan si la pantalla es angosta.
        # El PNG completo viaja en el mismo paquete (para Colab sin HTML, miniaturas y pruebas).
        import io as _io, base64 as _b64
        from matplotlib.transforms import Bbox as _Bbox
        for _num in list(plt.get_fignums()):
            fig = plt.figure(_num)
            try:
                grupos = {}
                for ax in fig.axes: grupos.setdefault(round(ax.get_position().x0, 2), []).append(ax)
                if len(grupos) != 2 or len({round(ax.get_position().y0, 2) for ax in fig.axes}) != 1: continue
                fig.canvas.draw(); r = fig.canvas.get_renderer()
                def _png(bb='tight', dpi=None):
                    b = _io.BytesIO(); fig.savefig(b, format='png', bbox_inches=bb, dpi=dpi); return _b64.b64encode(b.getvalue()).decode('ascii')
                partes = []
                for x0 in sorted(grupos):
                    bb = _Bbox.union([ax.get_tightbbox(r) for ax in grupos[x0]]).transformed(fig.dpi_scale_trans.inverted())
                    partes.append(_png(bb.padded(0.06), 144))
                html = ('<div style="display:flex;flex-wrap:wrap;gap:8px 16px;align-items:flex-start">'
                        + ''.join(f'<img alt="Esquema del punto (panel {i + 1} de 2)" src="data:image/png;base64,{p}" '
                                  'style="flex:1 1 300px;min-width:0;max-width:min(100%,420px);height:auto">' for i, p in enumerate(partes)) + '</div>')
                display({'text/html': html, 'image/png': _png(), 'text/plain': '<Figure>'}, raw=True)
                plt.close(fig)
            except Exception:
                pass
        return _orig(*a, **k)
    plt.show = _show_br; plt._dos_paneles_br = True
# --- fin pantalla angosta ---
try:
    from google.colab import output as _cxo
    _cxo.enable_custom_widget_manager()
except Exception:
    pass

# Registro global para la calificación formativa ponderada
try:
    REGISTRO_BR
except NameError:
    REGISTRO_BR = {}
def _registrar(id_preg, acierto, peso):
    REGISTRO_BR[id_preg] = (bool(acierto), float(peso))


def _num_coma(s):
    import re as _re
    # reemplaza punto decimal por coma en números dentro de un string ya formateado
    return _re.sub(r"(\d)\.(\d)", r"\1,\2", s)
def print_es(*args, **kwargs):
    args2=[_num_coma(a) if isinstance(a,str) else a for a in args]
    print(*args2, **kwargs)

try:
    PUNTO_DE_BR
except NameError:
    PUNTO_DE_BR = {}         # id_preg -> punto (botón «Mostrar solución») al que pertenece
    SOL_ANTES_BR = set()     # preguntas cuya solución se abrió antes del primer intento
    _PUNTO_ACTUAL_BR = [0]
def mostrar_solucion(func_solucion):
    # La solución es libre. Si se abre antes de intentar alguna pregunta del punto, queda anotado en la nota (no resta).
    _PUNTO_ACTUAL_BR[0] += 1; punto = _PUNTO_ACTUAL_BR[0]
    boton = Button(description="Mostrar solución", button_style='success', icon='eye')
    salida = Output()
    def _click(_):
        try: pend = [k for k, p in PUNTO_DE_BR.items() if p == punto and INTENTOS_BR.get(k, [0])[0] == 0]
        except NameError: pend = []
        SOL_ANTES_BR.update(pend)
        with salida:
            salida.clear_output()
            if pend:
                print("ℹ Abriste la solución antes de intentar " + ("una pregunta" if len(pend) == 1 else "varias preguntas") + " de este punto.")
                print("  No resta puntos: solo queda anotado en tu nota, para que sepas cuáles resolviste por tu cuenta.\n")
            func_solucion()
    boton.on_click(_click)
    display(boton, salida)

try:
    print_es
except NameError:
    def print_es(*args, **kwargs):
        import re as _re
        print(*[_re.sub(r"(\d)\.(\d)", r"\1,\2", a) if isinstance(a, str) else a for a in args], **kwargs)
# Signo menos tipográfico (−) en todo texto impreso: «-5» → «−5» (no toca guiones entre palabras, rangos ni 1e-3)
import builtins as _bi_br
if not getattr(_bi_br.print, '_menos_br', False):
    def _print_br(*a, _orig=_bi_br.print, **k):
        import re as _re
        _orig(*[_re.sub(r'(?<![\w.,\-])-(?=\d)', '−', x) if isinstance(x, str) else x for x in a], **k)
    _print_br._menos_br = True
    _bi_br.print = _print_br
# --- Cuestionarios Física BR v3: opciones barajadas, crédito por intento, nota sobre el total ---
import random as _rnd_br
try:
    PESOS_BR
except NameError:
    PESOS_BR = {}      # id_preg -> peso de cada pregunta mostrada
try:
    INTENTOS_BR
except NameError:
    INTENTOS_BR = {}   # id_preg -> [intentos, resuelta]
_CREDITO_BR = {1: 1.0, 2: 0.5, 3: 0.25}   # crédito según el intento en que se acierta
_MAX_INT_BR = 3

def _evaluar_intento(id_preg, ok, peso, explicacion, msg_ok, msg_mal):
    est = INTENTOS_BR.setdefault(id_preg, [0, False])
    if est[1]:
        print("✔ Ya resolviste esta pregunta; tu crédito quedó registrado.")
        if explicacion: print_es("→ " + explicacion)
        return
    if est[0] >= _MAX_INT_BR:
        print("⛔ Agotaste los 3 intentos con crédito: esta pregunta ya no suma.")
        if explicacion: print_es("→ " + explicacion)
        return
    est[0] += 1
    n = est[0]
    if ok:
        est[1] = True
        cred = _CREDITO_BR.get(n, 0.0)
        REGISTRO_BR[id_preg] = (cred, float(peso))
        print_es(msg_ok + f"  (intento {n}: crédito {int(cred*100)} %)")
        if explicacion: print_es("→ " + explicacion)
    else:
        REGISTRO_BR.setdefault(id_preg, (0.0, float(peso)))
        if n < _MAX_INT_BR:
            print(msg_mal + f"  Intento {n} de {_MAX_INT_BR}. Revisa tu razonamiento antes de volver a intentar.")
        else:
            print(msg_mal + " Agotaste los 3 intentos: esta pregunta ya no suma. Estudia la explicación:")
            if explicacion: print_es("→ " + explicacion)

def quiz_opcion_multiple(pregunta, opciones, indice_correcto, explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    orden = list(range(len(opciones))); _rnd_br.shuffle(orden)   # orden distinto en cada sesión
    correcta = orden.index(indice_correcto)
    letras = "abcdefgh"
    etiquetas = [f"{letras[i]}) {opciones[j]}" for i, j in enumerate(orden)]
    titulo = HTML(f"<b>{pregunta}</b>")
    radio  = widgets.RadioButtons(options=etiquetas, value=None, layout={'width':'max-content'})
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            if radio.index is None:
                print("⚠ Elige una opción."); return
            _evaluar_intento(id_preg, radio.index == correcta, peso, explicacion,
                             "✅ ¡Correcto!", "❌ Incorrecto.")
    boton.on_click(revisar); display(VBox([titulo, radio, boton, salida]))

def quiz_numerico(pregunta, valor_correcto, tolerancia_rel=0.05, unidad="", explicacion="", id_preg=None, peso=1.0):
    if id_preg is None: id_preg = f"_anon_{len(PESOS_BR)}"
    PESOS_BR[id_preg] = float(peso)
    PUNTO_DE_BR[id_preg] = _PUNTO_ACTUAL_BR[0]
    titulo = HTML(f"<b>{pregunta}</b>  <i>(tolerancia {tolerancia_rel*100:.0f}%)</i>")
    caja   = Text(placeholder=f"valor en {unidad}" if unidad else "tu respuesta")
    boton  = Button(description="Verificar", button_style='info'); salida = Output()
    def revisar(_):
        with salida:
            salida.clear_output()
            try: v = float(caja.value.strip().replace(",", "."))
            except ValueError: print("⚠ Escribe un número (usa coma o punto decimal)."); return
            ok = abs(v - valor_correcto) <= abs(valor_correcto)*tolerancia_rel
            _evaluar_intento(id_preg, ok, peso, explicacion,
                             f"✅ ¡Correcto!  ({valor_correcto:.3g} {unidad})", "❌ Fuera de rango.")
    boton.on_click(revisar); display(VBox([titulo, caja, boton, salida]))

def calificacion_final(total_preguntas=None):
    boton = Button(description="Calcular mi nota", button_style='primary', icon='star')
    salida = Output()
    def _click(_):
        with salida:
            salida.clear_output()
            if not PESOS_BR:
                print("No se encontraron cuestionarios en este taller."); return
            peso_total = sum(PESOS_BR.values())
            peso_ok = sum(float(c)*p for c, p in REGISTRO_BR.values())
            nota = 5.0*peso_ok/peso_total if peso_total > 0 else 0.0
            total = len(PESOS_BR)
            resp = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0])[0] > 0)
            prim = sum(1 for k in PESOS_BR if INTENTOS_BR.get(k, [0, False])[1] and INTENTOS_BR[k][0] == 1)
            print("="*46)
            print("          CALIFICACIÓN FORMATIVA")
            print("="*46)
            print(f"  Preguntas respondidas:       {resp} de {total}")
            print(f"  Acertadas al primer intento: {prim}")
            try: vistas = sum(1 for k in PESOS_BR if k in SOL_ANTES_BR)
            except NameError: vistas = 0
            print(f"  Solución vista antes de intentar: {vistas}")
            print_es(f"  Puntaje ponderado:           {peso_ok:.1f} / {peso_total:.1f}")
            print("-"*46)
            print(f"  NOTA:  {(f'%.1f'%nota).replace('.',',')}  / 5,0")
            print("="*46)
            if vistas: print("  ℹ En las preguntas donde abriste la solución antes, la nota no dice si sabes resolverlas sin ayuda: vuelve a intentarlas en unos días.")
            if resp < total: print(f"  ⚠ Faltan {total-resp} preguntas: cuentan como 0.")
            if nota >= 3.5: print("  ✅ ¡Buen trabajo! Dominas los conceptos.")
            elif nota >= 3.0: print("  🟡 Vas bien, repasa lo que fallaste.")
            else: print("  🔴 Repasa la teoría y vuelve a los puntos.")
            print("\n  Crédito por pregunta: 100 % al primer intento,")
            print("  50 % al segundo, 25 % al tercero. Predice y")
            print("  razona ANTES de verificar: adivinar cuesta nota.")
    boton.on_click(_click)
    display(VBox([HTML("<h3>📊 Tu calificación del taller</h3>"
        "<i>La nota se calcula sobre TODAS las preguntas del taller y premia acertar al primer intento.</i>"),
        boton, salida]))


# Calificación formativa del taller
calificacion_final(total_preguntas=16)
